In [1]:
# ==========================================
# BLOQUE 0: INSTALACIÓN Y CARGA DE DATOS
# ==========================================
!pip install pyspark
from pyspark.sql import SparkSession
from google.colab import files

print("Seleccione y suba el achivo csv:")
uploaded = files.upload()

spark = SparkSession.builder.appName("Nami_Casos_Separados").getOrCreate()


df_pedidos = spark.read.csv("pedidos.csv", header=True, inferSchema=True, sep=';')
rdd_pedidos = df_pedidos.rdd

print("\n--- DATOS CARGADOS Y LISTOS PARA LOS CASOS ---")
df_pedidos.show(5)

Seleccione y suba el achivo csv:


Saving pedidos.csv to pedidos.csv

--- DATOS CARGADOS Y LISTOS PARA LOS CASOS ---
+-------+----------+-------------+--------------+---------+----------------+--------------+---------+-----------+------------------+----------------------+-------------+
|sk_zona|sk_cliente|sk_repartidor|sk_restaurante|id_tiempo|id_pedido_origen|id_pago_origen|tipo_pago|monto_total|cantidad_productos|tiempo_entrega_minutos|estado_pedido|
+-------+----------+-------------+--------------+---------+----------------+--------------+---------+-----------+------------------+----------------------+-------------+
|      1|         1|            1|             1| 20251101|           PD001|         PG001|  Tarjeta|       25.5|                 1|                    40|    Entregado|
|      2|         2|            2|             2| 20251102|           PD002|         PG002|     Yape|       30.0|                 1|                    30|    Entregado|
|      3|         3|            3|             3| 20251103|         

In [ ]:
# ==========================================
# CASO 1: PEDIDOS CANCELADOS
# ==========================================
print("--- CASO 1: ANÁLISIS DE PEDIDOS CANCELADOS ---")

# Usamos la columna: "estado_pedido"
rdd_cancelados = rdd_pedidos.filter(lambda row: row["estado_pedido"] == "Cancelado")
total_cancelados = rdd_cancelados.count()
lista_cancelados = rdd_cancelados.collect()

print(f"Se han detectado {total_cancelados} pedidos cancelados en el sistema.")
print("Detalle de pérdidas:")
for pedido in lista_cancelados:
    # Usamos "id_pedido_origen", "sk_zona" y "monto_total"
    print(f"ID: {pedido['id_pedido_origen']} | Zona (SK): {pedido['sk_zona']} | Pérdida: S/{pedido['monto_total']}")

--- CASO 1: ANÁLISIS DE PEDIDOS CANCELADOS ---
ALERTA: Se han detectado 0 pedidos cancelados en el sistema.
Detalle de pérdidas:


In [ ]:
# ==========================================
# CASO 2: INGRESOS POR ZONA
# ==========================================
print("--- CASO 2: RANKING DE INGRESOS POR ZONA ---")

# Filtramos por "estado_pedido" y extraemos "sk_zona" y "monto_total"
rdd_ingresos_zona = rdd_pedidos.filter(lambda row: row["estado_pedido"] == "Entregado") \
                               .map(lambda row: (row["sk_zona"], row["monto_total"])) \
                               .reduceByKey(lambda a, b: a + b)

# Ordenamos de mayor a menor ingreso
resultados_finales = rdd_ingresos_zona.sortBy(lambda x: x[1], ascending=False).collect()

print("Ganancias totales por zona (SK):")
for zona, total in resultados_finales:
    print(f"Zona ID: {str(zona).ljust(10)} | Ingreso Total: S/{total:.2f}")


--- CASO 2: RANKING DE INGRESOS POR ZONA ---
Ganancias totales por zona (SK):
Zona ID: 2          | Ingreso Total: S/113.00
Zona ID: 3          | Ingreso Total: S/53.00
Zona ID: 1          | Ingreso Total: S/47.50
Zona ID: 4          | Ingreso Total: S/40.00


In [ ]:
# ==========================================
# CASO 3: TOP 3 REPARTIDORES
# ==========================================
print("--- CASO 3: RENDIMIENTO LOGÍSTICO (TOP REPARTIDORES) ---")

# Filtramos por "estado_pedido" y agrupamos por "sk_repartidor"
top_3_repartidores = rdd_pedidos.filter(lambda row: row["estado_pedido"] == "Entregado") \
                                  .map(lambda row: (row["sk_repartidor"], 1)) \
                                  .reduceByKey(lambda a, b: a + b) \
                                  .sortBy(lambda x: x[1], ascending=False) \
                                  .take(3)

print("LOS 3 MEJORES REPARTIDORES DE ÑAMI")
for posicion, (repartidor, cantidad) in enumerate(top_3_repartidores, 1):
    print(f"Puesto #{posicion} | Repartidor ID: {str(repartidor).ljust(10)} | Entregas completadas: {cantidad}")

--- CASO 3: RENDIMIENTO LOGÍSTICO (TOP REPARTIDORES) ---
🏆 LOS 3 MEJORES REPARTIDORES DE ÑAMI 🏆
Puesto #1 | Repartidor ID: 2          | Entregas completadas: 3
Puesto #2 | Repartidor ID: 3          | Entregas completadas: 2
Puesto #3 | Repartidor ID: 1          | Entregas completadas: 1


In [ ]:
# ==========================================
# BLOQUE 4: SPARK SQL - PREFERENCIAS Y DEMANDA (CORREGIDO)
# ==========================================
print("--- CASO 4: ANÁLISIS DE DEMANDA CON SPARK SQL ---")

# 1. Crear una Vista Temporal SQL a partir del DataFrame
df_pedidos.createOrReplaceTempView("vista_dw_pedidos")

# 2. Escribimos nuestra consulta en SQL usando TRY_CAST para evitar errores con textos "NULL"
consulta_sql = """
    SELECT
        tipo_pago AS Metodo_Pago,
        COUNT(id_pedido_origen) AS Total_Pedidos,
        ROUND(SUM(TRY_CAST(monto_total AS DOUBLE)), 2) AS Ingresos_Totales,
        ROUND(AVG(TRY_CAST(tiempo_entrega_minutos AS DOUBLE)), 2) AS Tiempo_Promedio_Minutos
    FROM vista_dw_pedidos
    WHERE estado_pedido = 'Entregado'
    GROUP BY tipo_pago
    ORDER BY Ingresos_Totales DESC
"""

# 3. Ejecutar y mostrar resultados
resultados_sql = spark.sql(consulta_sql)
print("Análisis de ingresos y tiempos logísticos según el método de pago:")
resultados_sql.show()

--- CASO 4: ANÁLISIS DE DEMANDA CON SPARK SQL ---
Análisis de ingresos y tiempos logísticos según el método de pago:
+-----------+-------------+----------------+-----------------------+
|Metodo_Pago|Total_Pedidos|Ingresos_Totales|Tiempo_Promedio_Minutos|
+-----------+-------------+----------------+-----------------------+
|    Tarjeta|            4|           145.5|                  46.25|
|       Yape|            4|            90.0|                  33.33|
|       Plin|            1|            18.0|                   30.0|
+-----------+-------------+----------------+-----------------------+



In [ ]:
# ==========================================
# BLOQUE 5: ESTADÍSTICA AVANZADA E INCIDENCIAS (CORREGIDO DEFINITIVO)
# ==========================================
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import pyspark.sql.functions as F

print("--- CASO 5: ESTADÍSTICA AVANZADA Y DETECCIÓN DE ANOMALÍAS ---")

# 1. SOLUCIÓN AL ERROR: Usamos TRY_CAST igual que en SQL.
# Esto convierte el texto a número de forma segura, si ve la palabra "NULL",
# no eliminar, solo la convierte en un vacío real que Spark síi entiende
df_stats = df_pedidos.withColumn("cantidad_productos", F.expr("TRY_CAST(cantidad_productos AS DOUBLE)")) \
                     .withColumn("tiempo_entrega_minutos", F.expr("TRY_CAST(tiempo_entrega_minutos AS DOUBLE)"))

# 2. Limpiamos las filas que quedaron como vacios reales
df_stats = df_stats.na.drop(subset=["cantidad_productos", "tiempo_entrega_minutos"])

# ---------------------------------------------------------
# PARTE A: CORRELACION DE PEARSON
# ---------------------------------------------------------
print("\n[A] MATRIZ DE CORRELACIÓN:")
print("¿Pedir más productos significa que el pedido tardará más en llegar?")

# Preparamos las columnas para el modelo matematico
assembler = VectorAssembler(inputCols=["cantidad_productos", "tiempo_entrega_minutos"], outputCol="features")
df_vector = assembler.transform(df_stats)

# Calculamos la correlacion
matriz_corr = Correlation.corr(df_vector, "features").head()[0]
print(str(matriz_corr).replace('DenseMatrix', 'Resultado de la Matriz:\n'))
print("\n* Nota: Un valor cercano a 1 indica alta relación; cercano a 0 indica que no tienen relación directa.")

# ---------------------------------------------------------
# PARTE B: DETECCION DE OUTLIERS (VALORES ATIPICOS EN TIEMPO)
# ---------------------------------------------------------
print("\n[B] DETECCIÓN DE INCIDENCIAS LOGÍSTICAS (OUTLIERS):")

# Calculamos los cuartiles y el Rango Intercuartílico (IQR) matematicamente
cuantiles = df_stats.approxQuantile("tiempo_entrega_minutos", [0.25, 0.75], 0.0)
Q1 = cuantiles[0]
Q3 = cuantiles[1]
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

print(f"-> Según nuestra estadística, cualquier pedido que tarde más de {limite_superior} minutos es una ANOMALÍA.")

# Filtramos para encontrar los pedidos criticos
outliers = df_stats.filter(F.col("tiempo_entrega_minutos") > limite_superior)
total_anomalias = outliers.count()

print(f"Se detectaron {total_anomalias} pedidos con tiempos inaceptables.")
outliers.select("id_pedido_origen", "sk_repartidor", "cantidad_productos", "tiempo_entrega_minutos", "estado_pedido").show(5)

--- CASO 5: ESTADÍSTICA AVANZADA Y DETECCIÓN DE ANOMALÍAS ---

[A] MATRIZ DE CORRELACIÓN:
¿Pedir más productos significa que el pedido tardará más en llegar?
Resultado de la Matriz:
([[1.        , 0.30714756],
             [0.30714756, 1.        ]])

* Nota: Un valor cercano a 1 indica alta relación; cercano a 0 indica que no tienen relación directa.

[B] DETECCIÓN DE INCIDENCIAS LOGÍSTICAS (OUTLIERS):
-> Según nuestra estadística, cualquier pedido que tarde más de 67.5 minutos es una ANOMALÍA.
¡ALERTA CRÍTICA! Se detectaron 0 pedidos con tiempos inaceptables.
+----------------+-------------+------------------+----------------------+-------------+
|id_pedido_origen|sk_repartidor|cantidad_productos|tiempo_entrega_minutos|estado_pedido|
+----------------+-------------+------------------+----------------------+-------------+
+----------------+-------------+------------------+----------------------+-------------+

